In [ ]:
from pathlib import Path
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20260819
random.seed(SEED); np.random.seed(SEED)
warnings.filterwarnings('ignore')
DATA_ROOT = Path('TERA Analysis')
PROJECT_ROOT = Path('TERA')
RESULTS = PROJECT_ROOT / 'results'; FIGURES = PROJECT_ROOT / 'figures'
RESULTS.mkdir(exist_ok=True); FIGURES.mkdir(exist_ok=True)
print('Data:', DATA_ROOT)
print('Outputs:', PROJECT_ROOT)

In [ ]:
from matplotlib.lines import Line2D
raw=pd.read_csv(DATA_ROOT/'Data/EDM_raw_data.csv')
selected=raw[(raw.USUBJID.isin([101005,102009])) & (raw.PARAMCD=='ACTUAL')].copy()
selected['opening_datetime']=pd.to_datetime(selected.AVAL,errors='coerce')
selected['date']=pd.to_datetime(selected.ADT,errors='coerce')
selected=selected[selected.opening_datetime.notna()].copy()
selected['opening_hour']=selected.opening_datetime.dt.hour+selected.opening_datetime.dt.minute/60
selected['day_type']=np.where(selected.date.dt.dayofweek>=5,'Weekend','Weekday')
selected['participant']=selected.USUBJID.map({101005:'Participant A',102009:'Participant B'})
figure_data=selected.rename(columns={'ADY':'study_day'})[['participant','study_day','opening_hour','day_type']].sort_values(['participant','study_day'])
figure_data.to_csv(RESULTS/'figure2_deidentified_data.csv',index=False)
display(figure_data.groupby('participant').agg(n=('study_day','size'),first_day=('study_day','min'),last_day=('study_day','max')))


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.family':'serif','font.weight':'semibold','axes.labelsize':20,
                     'axes.titlesize':20,'xtick.labelsize':16,'ytick.labelsize':16,
                     'axes.linewidth':2,'axes.edgecolor':'black','legend.fontsize':16})
colors={'Weekday':'#2166AC','Weekend':'#C62828'}
panels=[('Participant A','(a) Participant A'),('Participant B','(b) Participant B')]
fig,axes=plt.subplots(2,1,figsize=(12,7.5),sharex=True,sharey=True,constrained_layout=True)
for ax,(participant,title) in zip(axes,panels):
    part=figure_data[figure_data.participant==participant]
    for kind in ['Weekday','Weekend']:
        pts=part[part.day_type==kind]
        ax.scatter(pts.study_day,pts.opening_hour,s=18,color=colors[kind],edgecolors='white',linewidths=.25,label=kind)
    ax.set_title(title,loc='center',fontweight='semibold'); ax.set_ylim(0,24)
    ax.set_yticks([0,6,12,18,24],labels=['00:00','06:00','12:00','18:00','24:00'])
    ax.grid(axis='y',color='#D9D9D9',linewidth=.7); ax.grid(axis='x',color='#EEEEEE',linewidth=.5)
fig.supylabel('Bottle-opening time',fontweight='semibold'); axes[-1].set_xlabel('Study day',fontweight='semibold')
legend=[Line2D([0],[0],marker='o',linestyle='none',label=k,markerfacecolor=v,markeredgecolor='white',markersize=8) for k,v in colors.items()]
fig.legend(handles=legend,loc='upper center',ncol=2,frameon=False,bbox_to_anchor=(.5,1.03))
fig.savefig(FIGURES/'figure2_dynamic_medication_patterns.png',dpi=300,bbox_inches='tight')
fig.savefig(FIGURES/'figure2_dynamic_medication_patterns.tiff',dpi=300,bbox_inches='tight',pil_kwargs={'compression':'tiff_lzw'})
plt.show()
